# H6: score v3-r16 (HF v4) on v4-mixed-r16's YouTube-only test slice

SESSIONS.md **H6** -- the branch gate for everything else left in the plan
(H2b, H4b, D1/D2/D3). Every gate tier compares a candidate only against its
own base-model baseline; nothing has ever scored v3-r16 (published as
`winhsss/Reworkwhisper-large-v4`) on the exact **228 `source: youtube`
segments** of v4-mixed-r16's (`winhsss/Reworkwhisper-large-v5`)
tier1_in_domain test split -- v5's own CER there is **0.0764**, retention
**0.7094** (874 candidates, 620 retained -- computed from this repo's own
`Outputs/v4-mixed-r16/audit/predictions_tier1_in_domain.csv`, not a number
from SESSIONS.md). This is the one comparison that is real speech,
human-reviewed, AND same-domain as the production audio under investigation.

**GPU required (T4 is enough -- inference only, no training).**

**Before running:**
- Commit and push `scripts/eval_v3_on_youtube_test.py` to GitHub `main` first
  -- Cell 1 clones `main`, not your local tree.
- Attach as Kaggle Dataset inputs (Add Data):
  - the **`youtube-meetings`** dataset (the same one `build_mixed_dataset`
    used to train v4-mixed-r16). The 228 test segments' audio lives at
    `<mount>/audio/<meeting_id>/seg_XXXX.wav` -- identical relative layout to
    what `mixed-noisy-v1` copied in, so the merged dataset itself is not
    needed for this notebook.
  - a **new dataset you zip and upload**, holding these three paths from this
    repo's `Outputs/` (gitignored, never pushed to GitHub -- preserve the
    relative layout shown so the defaults below need only a mount-path edit):
    - `v3-r16/checkpoints/best/` (111 MB -- the raw LoRA adapter, before any
      lambda is baked in)
    - `v3-r16/config.json`
    - `v4-mixed-r16/validated_manifest.jsonl` (3 MB -- already resolved and
      split-assigned, used to select the exact 228 segments without
      recomputing `resolve_splits` here)

**Disk:** ~3.1 GB base model download into `~/.cache/huggingface`. Nothing
else is written except the predictions CSV.

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the paths below -- Kaggle slugs
the dataset name, so this can differ from what you expect (mount paths nest
one level deeper than the attached dataset name, e.g.
`/kaggle/input/datasets/<user>/<slug>/<slug>`).

In [ ]:
!ls -la /kaggle/input
!ls -la /kaggle/input/datasets/*/*

## 3. Set platform-specific paths

Edit these three to match what the cell above printed. This is the only
place a `/kaggle/input/...` path is written -- `scripts/eval_v3_on_youtube_test.py`
never hardcodes one.

In [ ]:
AUDIO_ROOT = "/kaggle/input/datasets/<user>/youtube-meetings/youtube-meetings/audio"  # edit
H6_ARTIFACTS = "/kaggle/input/datasets/<user>/h6-artifacts/h6-artifacts"              # edit -- the zip from the intro cell

RUN_DIR = f"{H6_ARTIFACTS}/v3-r16"                                    # config.json + checkpoints/best/
MANIFEST = f"{H6_ARTIFACTS}/v4-mixed-r16/validated_manifest.jsonl"

print("AUDIO_ROOT exists:", os.path.exists(AUDIO_ROOT))
print("RUN_DIR/config.json exists:", os.path.exists(f"{RUN_DIR}/config.json"))
print("RUN_DIR/checkpoints/best exists:", os.path.exists(f"{RUN_DIR}/checkpoints/best"))
print("MANIFEST exists:", os.path.exists(MANIFEST))

## 4. Environment

No `HF_TOKEN` needed -- this is inference only, nothing is pushed.

In [ ]:
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

## 5. Run the eval

Selects the 228 `split: test, source: youtube` records from `MANIFEST`, loads
`vinai/PhoWhisper-large` + v3-r16's raw checkpoint (pre-lambda-bake), scales
to **lambda=0.5** in memory (`src.lora.set_lambda` -- the same call
`stage_sweep_gate` makes for every lambda in a sweep), transcribes, and
writes `predictions_v3-r16_youtube228.csv`.

**Untested on the author's machine** (no torch/GPU there) -- per CLAUDE.md's
own "Kaggle code never works first try", expect to fix at least one thing
here before it runs clean.

In [ ]:
import subprocess

OUT_CSV = "/kaggle/working/predictions_v3-r16_youtube228.csv"

cmd = [
    "python", "-m", "scripts.eval_v3_on_youtube_test",
    "--manifest", MANIFEST,
    "--audio-root", AUDIO_ROOT,
    "--run-dir", RUN_DIR,
    "--lam", "0.5",
    "--out", OUT_CSV,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise SystemExit(f"eval_v3_on_youtube_test failed with exit code {proc.returncode}")

## 6. Read the result -- what it decides

Compare the printed `cer=` line against v4-mixed-r16's **0.0764** on this
same 228-segment slice (this notebook's intro cell). This is a branch gate,
not a nice-to-know number:

- v3-r16 CER **lower** than 0.0764 -> v5 is a real regression on production's
  domain; H4(b), D1 (casing recovery), D3 (mix-ratio retrain) all have a
  basis to proceed on.
- v3-r16 CER **higher** than 0.0764 -> v5 is actually *better* on production's
  domain, the module-8 symptom (`team`->`tim`, dropped job titles) is a
  casing-convention artifact rather than a transcription-quality drop, and D3
  loses most of its reason to exist (SESSIONS.md's stop-gate 4).

Do not read this by comparing either number to the PhoWhisper baseline -- the
only comparison that means anything here is v3-r16 vs v4-mixed-r16.

`predictions_v3-r16_youtube228.csv` is written to `/kaggle/working/` --
download it (or the printed `english_token_retention`) before the session
ends; Kaggle does not persist `/working` across sessions unless saved as a
Version output.